# 06 · Models on the enriched features (LightGBM x3, XGBoost, CatBoost, logistic regression)

Trains the same families again on `features3.parquet` (the version with trend and severity features) and saves their out-of-fold and test predictions for stacking in notebook 13.

**Needs:** `features3.parquet` (03) and `search.json` (04, supplies the three best LightGBM settings). **Writes:** `n_lgb0/1/2`, `n_xgb`, `n_cat`, `n_lr` (each `_oof.npy` and `_test.npy`). **Time:** about 40 minutes.

Observed out-of-fold AUC: LightGBM 0.7515 / 0.7512 / 0.7515, XGBoost 0.7510, CatBoost 0.7481, logistic regression 0.7428.

**Setup.** Install the libraries once (uncomment), then set the two paths in the next cell.

In [ ]:
# %pip install -q lightgbm xgboost scikit-learn pandas pyarrow scipy

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# ---- EDIT THESE TWO PATHS --------------------------------------------------
DATA = os.environ.get('KAGGLE_DATA', 'data')   # folder holding the unzipped competition CSVs (train.csv, test.csv, patients.csv, ...)
WORK = os.environ.get('KAGGLE_WORK', 'work')   # folder where features, predictions and submissions are saved (shared by all notebooks)
# -----------------------------------------------------------------------------
Path(WORK).mkdir(parents=True, exist_ok=True)
print('DATA ->', Path(DATA).resolve()); print('WORK ->', Path(WORK).resolve())

In [ ]:
import pandas as pd, numpy as np, lightgbm as lgb, xgboost as xgb, json, sys, time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def main(part):
    W=WORK; D=DATA
    drop=("hrs_","visits_","n_transfers","first_unit","last_unit","icu_","had_ed","ward_after","proc_","n_proc")
    X=pd.read_parquet(f'{W}/features3.parquet'); tr=pd.read_csv(f'{D}/train.csv'); te=pd.read_csv(f'{D}/test.csv')
    catc=[c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c]) and c!='encounter_id']
    Xg=X.copy()
    for c in catc: Xg[c]=Xg[c].astype('category')
    T=Xg.merge(tr,on='encounter_id'); S=te.merge(Xg,on='encounter_id',how='left'); y=T.readmit_30d.values
    assert S.encounter_id.tolist()==te.encounter_id.tolist()
    feats=[c for c in T.columns if c not in('encounter_id','readmit_30d') and not c.startswith(drop)]
    folds=list(StratifiedKFold(5,shuffle=True,random_state=0).split(T,y)); t0=time.time()
    def save(name,o,t): np.save(f'{W}/n_{name}_oof.npy',o); np.save(f'{W}/n_{name}_test.npy',t); print(name,'OOF AUC %.5f'%roc_auc_score(y,o),'t=%.0f'%(time.time()-t0),flush=True)
    if part=='lgb':
        res=json.load(open(f'{W}/search.json'))[:3]; base=dict(objective='binary',learning_rate=0.05,bagging_freq=1,verbose=-1,n_jobs=2,cat_smooth=50,max_cat_to_onehot=8)
        for i,(auc,q,its) in enumerate(res):
            p={**base,**q}; o=np.zeros(len(T)); t=np.zeros(len(S)); NS=2
            for f,(a,b) in enumerate(folds):
                for s in range(NS):
                    p['seed']=100*s+f; m=lgb.train(p,lgb.Dataset(T.loc[a,feats],y[a]),5000,valid_sets=[lgb.Dataset(T.loc[b,feats],y[b])],callbacks=[lgb.early_stopping(200,verbose=False)])
                    o[b]+=m.predict(T.loc[b,feats],num_iteration=m.best_iteration)/NS; t+=m.predict(S[feats],num_iteration=m.best_iteration)/(5*NS)
            save(f'lgb{i}',o,t)
    if part=='xgb':
        P=dict(objective='binary:logistic',eval_metric='auc',tree_method='hist',max_depth=3,eta=0.04,min_child_weight=20,subsample=0.8,colsample_bytree=0.3,reg_lambda=20,max_bin=63,max_cat_to_onehot=8,nthread=2)
        dS=xgb.DMatrix(S[feats],enable_categorical=True); o=np.zeros(len(T)); t=np.zeros(len(S))
        for f,(a,b) in enumerate(folds):
            da=xgb.DMatrix(T.loc[a,feats],y[a],enable_categorical=True); db=xgb.DMatrix(T.loc[b,feats],y[b],enable_categorical=True)
            m=xgb.train({**P,'seed':f},da,4000,evals=[(db,'v')],early_stopping_rounds=200,verbose_eval=False); n=m.best_iteration+1
            o[b]=m.predict(db,iteration_range=(0,n)); t+=m.predict(dS,iteration_range=(0,n))/5
        save('xgb',o,t)
    if part=='cat':
        from catboost import CatBoostClassifier, Pool
        cats=[c for c in feats if c in catc]; Tc=T[feats].copy(); Sc=S[feats].copy()
        for c in cats: Tc[c]=Tc[c].astype(object).fillna('NA').astype(str); Sc[c]=Sc[c].astype(object).fillna('NA').astype(str)
        o=np.zeros(len(T)); t=np.zeros(len(S))
        for f,(a,b) in enumerate(folds):
            m=CatBoostClassifier(iterations=3000,learning_rate=0.06,depth=4,l2_leaf_reg=10,rsm=0.3,border_count=64,eval_metric='AUC',od_type='Iter',od_wait=200,thread_count=2,verbose=0,random_seed=f)
            m.fit(Pool(Tc.iloc[a],y[a],cat_features=cats),eval_set=Pool(Tc.iloc[b],y[b],cat_features=cats),use_best_model=True)
            o[b]=m.predict_proba(Pool(Tc.iloc[b],cat_features=cats))[:,1]; t+=m.predict_proba(Pool(Sc,cat_features=cats))[:,1]/5
        save('cat',o,t)
    if part=='lr':
        num=[c for c in feats if pd.api.types.is_numeric_dtype(T[c]) and T[c].notna().sum()>0 and T[c].nunique()>1]; cat=[c for c in feats if c in catc]
        A=pd.concat([T[num],S[num]]); miss=A.columns[A.isna().mean()>0.01]; lo,hi=T[num].quantile(.005),T[num].quantile(.995); med=T[num].clip(lo,hi,axis=1).median()
        prep=lambda df: pd.concat([df[num].clip(lo,hi,axis=1).fillna(med),df[miss].isna().astype(int).add_suffix('_na'),df[cat].astype(object).fillna('NA')],axis=1)
        Z=pd.get_dummies(pd.concat([prep(T),prep(S)],ignore_index=True),columns=cat,dtype=float).astype(np.float32); Zt,Zs=Z.iloc[:len(T)].values,Z.iloc[len(T):].values
        o=np.zeros(len(T)); t=np.zeros(len(S))
        for a,b in folds:
            sc=StandardScaler().fit(Zt[a]); m=LogisticRegression(C=0.02,max_iter=500).fit(sc.transform(Zt[a]),y[a]); o[b]=m.predict_proba(sc.transform(Zt[b]))[:,1]; t+=m.predict_proba(sc.transform(Zs))[:,1]/5
        save('lr',o,t)

In [ ]:
for part in ['lr', 'cat', 'xgb', 'lgb']:   # 'lgb' is the slow one (about 25 min)
    main(part)